# CORS 문제 교과서 수준 분석

## 1장: CORS 기본 개념

### Same-Origin Policy란 무엇인가

**Same-Origin Policy(동일 출처 정책)**는 브라우저의 핵심 보안 메커니즘입니다. 이 정책은 한 출처(origin)에서 로드된 문서나 스크립트가 다른 출처의 리소스와 상호작용하는 것을 제한합니다.

**Origin의 구성 요소:**
- 프로토콜 (Protocol): `http://` 또는 `https://`
- 도메인 (Domain): `kotlip.kr`
- 포트 (Port): `443` (HTTPS 기본), `80` (HTTP 기본)

**예시:**
```
https://kotlip.kr/api/video/upload/status
  ↓
프로토콜: https
도메인: kotlip.kr
포트: 443 (기본값, 생략됨)
```

**Same-Origin 판단:**
- ✅ `https://kotlip.kr` ↔ `https://kotlip.kr/api/...` → **Same-Origin**
- ❌ `https://kotlip.kr` ↔ `https://proxy1.aiserv.ktcloud.com:10285` → **Cross-Origin**

### Cross-Origin 요청이 필요한 이유

현대 웹 애플리케이션은 여러 서버와 통신합니다:
- 프론트엔드 서버: `https://kotlip.kr` (사용자에게 UI 제공)
- API 서버: `https://proxy1.aiserv.ktcloud.com:10285` (데이터 처리)

프론트엔드에서 API 서버로 직접 요청을 보내려면 **Cross-Origin 요청**이 필요합니다.

### CORS가 해결하는 문제

**CORS (Cross-Origin Resource Sharing)**는 브라우저가 Cross-Origin 요청을 안전하게 허용할 수 있도록 하는 메커니즘입니다.

**CORS 없이 발생하는 문제:**
```
프론트엔드 (kotlip.kr) → API 서버 (proxy1.aiserv.ktcloud.com)
  ↓
브라우저: "Cross-Origin 요청은 차단합니다!"
  ↓
❌ CORS 오류 발생
```

**CORS로 해결:**
```
프론트엔드 (kotlip.kr) → API 서버
  ↓
API 서버: "kotlip.kr에서 온 요청을 허용합니다" (CORS 헤더 포함)
  ↓
✅ 브라우저가 요청 허용
```

### 브라우저의 CORS 검증 시점

브라우저는 **응답을 받은 후** CORS 검증을 수행합니다:

1. **요청 전송**: 브라우저가 요청을 서버로 전송
2. **서버 응답**: 서버가 데이터와 함께 CORS 헤더를 포함하여 응답
3. **CORS 검증**: 브라우저가 응답 헤더를 검사
4. **결과 처리**:
   - 검증 통과 → JavaScript에 응답 전달
   - 검증 실패 → CORS 오류 발생, 응답 차단

**중요**: 서버는 요청을 처리하고 응답을 보내지만, 브라우저가 최종적으로 응답을 JavaScript에 전달할지 결정합니다.

---

## 2장: CORS 헤더 상세 분석

### Access-Control-Allow-Origin의 의미와 동작

`Access-Control-Allow-Origin` 헤더는 서버가 어떤 출처(origin)의 요청을 허용하는지 브라우저에게 알려줍니다.

#### 와일드카드 `*`의 사용 조건

**형식:**
```
Access-Control-Allow-Origin: *
```

**의미**: 모든 출처에서 온 요청을 허용합니다.

**제한 사항:**
- ✅ **단순 요청** (Simple Request)에만 사용 가능
- ❌ **credentials 모드** (`credentials: 'include'`)와 함께 사용 불가
- ❌ **인증 정보** (쿠키, Authorization 헤더)가 필요한 요청과 함께 사용 불가

**예시 - 와일드카드 사용 가능:**
```javascript
// credentials 없이 요청
fetch('https://api.example.com/data')
  // credentials 기본값: 'same-origin'
```

**예시 - 와일드카드 사용 불가:**
```javascript
// credentials: 'include' 사용
fetch('https://api.example.com/data', {
  credentials: 'include'  // 쿠키 포함
})
// ❌ Access-Control-Allow-Origin: * 와 충돌!
```

#### 특정 origin 지정 방식

**형식:**
```
Access-Control-Allow-Origin: https://kotlip.kr
```

**의미**: 오직 `https://kotlip.kr`에서 온 요청만 허용합니다.

**장점:**
- ✅ credentials 모드와 함께 사용 가능
- ✅ 보안성 향상 (특정 출처만 허용)
- ✅ 인증 정보 전송 가능

**동적 origin 허용:**
서버는 요청의 `Origin` 헤더를 확인하여 동적으로 허용할 origin을 결정할 수 있습니다:

```
요청 헤더:
  Origin: https://kotlip.kr

서버 응답:
  Access-Control-Allow-Origin: https://kotlip.kr
```

### Access-Control-Allow-Credentials의 역할

#### credentials 모드 (`credentials: 'include'`)란

**credentials 모드**는 Cross-Origin 요청 시 쿠키와 인증 정보를 함께 전송하는 방식입니다.

**JavaScript 예시:**
```javascript
fetch('https://api.example.com/data', {
  credentials: 'include'  // 쿠키 포함하여 전송
})
```

**전송되는 정보:**
- 쿠키 (Cookies)
- HTTP 인증 정보 (Authorization 헤더)
- 클라이언트 인증서

#### 쿠키, 인증 헤더 전송과의 관계

**credentials 없이:**
```
요청:
  GET /api/data HTTP/1.1
  Origin: https://kotlip.kr
  (쿠키 없음)

응답:
  Access-Control-Allow-Origin: *
  (데이터만 전송)
```

**credentials: 'include' 사용:**
```
요청:
  GET /api/data HTTP/1.1
  Origin: https://kotlip.kr
  Cookie: session_id=abc123  ← 쿠키 포함!

응답:
  Access-Control-Allow-Origin: https://kotlip.kr  ← 와일드카드 불가!
  Access-Control-Allow-Credentials: true
```

### 두 헤더의 상호작용 규칙

#### 와일드카드와 credentials의 호환성 문제

**CORS 정책 규칙:**
> `Access-Control-Allow-Origin: *`와 `Access-Control-Allow-Credentials: true`는 **동시에 사용할 수 없습니다.**

**이유:**
- 와일드카드 `*`는 "모든 출처 허용"을 의미
- credentials는 "인증 정보 전송"을 의미
- 보안상 모든 출처에 인증 정보를 허용하는 것은 위험

**올바른 조합:**

| Access-Control-Allow-Origin | Access-Control-Allow-Credentials | 사용 가능? |
|------------------------------|----------------------------------|-----------|
| `*` | 없음 | ✅ 가능 |
| `*` | `true` | ❌ **불가능** |
| `https://kotlip.kr` | 없음 | ✅ 가능 |
| `https://kotlip.kr` | `true` | ✅ 가능 |

**현재 문제 상황:**
```
백엔드 응답:
  Access-Control-Allow-Origin: *
  Access-Control-Allow-Credentials: true

프론트엔드 요청:
  credentials: 'include'

결과:
  ❌ 브라우저가 CORS 정책 위반으로 차단
```

---

## 3장: nginx 설정 분석

### $http_origin 변수의 의미

#### 요청 헤더에서 Origin 추출

nginx의 `$http_origin` 변수는 클라이언트가 보낸 요청 헤더의 `Origin` 값을 자동으로 추출합니다.

**요청 예시:**
```
GET /api/video/upload/status HTTP/1.1
Host: kotlip.kr
Origin: https://kotlip.kr  ← 이 값을 $http_origin이 추출
```

**nginx 설정:**
```nginx
add_header Access-Control-Allow-Origin $http_origin always;
```

**동작:**
- 요청의 `Origin: https://kotlip.kr` → `$http_origin = "https://kotlip.kr"`
- nginx가 응답에 `Access-Control-Allow-Origin: https://kotlip.kr` 추가

#### 동적 origin 허용 방식

`$http_origin`을 사용하면 요청한 origin을 그대로 허용할 수 있습니다:

**장점:**
- 여러 도메인에서 요청이 와도 각각 허용 가능
- credentials 모드와 호환
- 보안성 유지 (요청한 origin만 허용)

**예시:**
```
요청 1:
  Origin: https://kotlip.kr
  → Access-Control-Allow-Origin: https://kotlip.kr

요청 2:
  Origin: https://kotlip.co.kr
  → Access-Control-Allow-Origin: https://kotlip.co.kr
```

### proxy_hide_header와 add_header의 동작

#### 백엔드 헤더 숨기기

**nginx 설정:**
```nginx
proxy_hide_header Access-Control-Allow-Origin;
proxy_hide_header Access-Control-Allow-Credentials;
```

**의미:**
- 백엔드가 보낸 `Access-Control-Allow-Origin` 헤더를 응답에서 제거
- 백엔드가 보낸 `Access-Control-Allow-Credentials` 헤더를 응답에서 제거

**이유:**
- 백엔드가 `Access-Control-Allow-Origin: *`를 보내는 경우
- nginx가 이를 숨기고 올바른 헤더로 교체해야 함

**흐름:**
```
백엔드 응답:
  Access-Control-Allow-Origin: *
  Access-Control-Allow-Credentials: true
  (데이터)

nginx 처리:
  proxy_hide_header로 위 헤더 제거
  ↓
  add_header로 새 헤더 추가
  ↓
최종 응답:
  Access-Control-Allow-Origin: https://kotlip.kr
  Access-Control-Allow-Credentials: true
  (데이터)
```

#### nginx 헤더 추가 메커니즘

**nginx 설정:**
```nginx
add_header Access-Control-Allow-Origin $http_origin always;
add_header Access-Control-Allow-Credentials true always;
```

**동작:**
1. `add_header`: 응답에 새 헤더 추가
2. `$http_origin`: 요청의 Origin 값을 사용
3. `always`: 성공/에러 응답 모두에 헤더 추가

**always 플래그의 역할:**
- 기본적으로 `add_header`는 성공 응답(2xx, 3xx)에만 헤더 추가
- `always`를 사용하면 에러 응답(4xx, 5xx)에도 헤더 추가
- CORS 오류를 방지하기 위해 필요

**예시:**
```nginx
# always 없이
location /api {
    add_header Access-Control-Allow-Origin $http_origin;
    # 200 OK 응답에만 헤더 추가
    # 404, 500 등에는 헤더 없음 → CORS 오류 가능
}

# always 사용
location /api {
    add_header Access-Control-Allow-Origin $http_origin always;
    # 모든 응답에 헤더 추가
    # 404, 500에도 헤더 포함 → CORS 오류 방지
}
```

---

## 4장: 브라우저의 CORS 검증 과정

### Preflight 요청 (OPTIONS) 처리

**Preflight 요청**은 브라우저가 실제 요청 전에 서버의 CORS 정책을 확인하는 사전 요청입니다.

**Preflight가 발생하는 조건:**
- 복잡한 요청 (Complex Request)
  - PUT, DELETE, PATCH 메서드
  - 커스텀 헤더 사용
  - Content-Type이 `application/json` 등

**Preflight 요청 예시:**
```
OPTIONS /api/video/upload/status HTTP/1.1
Origin: https://kotlip.kr
Access-Control-Request-Method: GET
Access-Control-Request-Headers: authorization
```

**서버 응답 (nginx 설정):**
```nginx
if ($request_method = OPTIONS) {
    return 204;
}
```

**응답 헤더:**
```
HTTP/1.1 204 No Content
Access-Control-Allow-Origin: https://kotlip.kr
Access-Control-Allow-Credentials: true
Access-Control-Allow-Methods: GET, POST, PUT, PATCH, DELETE, OPTIONS
Access-Control-Allow-Headers: accept, authorization, content-type, ...
```

**브라우저 검증:**
- Preflight 응답이 올바르면 → 실제 요청 전송
- Preflight 응답이 부족하면 → CORS 오류, 실제 요청 차단

### 실제 요청 시 검증 단계

브라우저는 실제 요청의 응답을 받은 후 다음 단계로 검증합니다:

#### 1. Origin 헤더 확인

**요청 헤더:**
```
GET /api/video/upload/status HTTP/1.1
Origin: https://kotlip.kr
```

브라우저가 자동으로 현재 페이지의 origin을 `Origin` 헤더에 추가합니다.

#### 2. Access-Control-Allow-Origin 검증

**응답 헤더 확인:**
```
Access-Control-Allow-Origin: https://kotlip.kr
```

**검증 규칙:**
- ✅ 응답의 `Access-Control-Allow-Origin`이 요청의 `Origin`과 일치 → 통과
- ✅ 응답의 `Access-Control-Allow-Origin`이 `*` → 통과 (단, credentials 없을 때만)
- ❌ 응답에 `Access-Control-Allow-Origin` 없음 → 실패
- ❌ 응답의 `Access-Control-Allow-Origin`이 요청의 `Origin`과 불일치 → 실패

#### 3. credentials 모드일 때 추가 검증

**요청:**
```javascript
fetch('https://api.example.com/data', {
  credentials: 'include'
})
```

**검증 규칙:**
- ✅ `Access-Control-Allow-Origin`이 특정 origin (와일드카드 아님)
- ✅ `Access-Control-Allow-Credentials: true` 존재
- ❌ `Access-Control-Allow-Origin: *` → **즉시 실패**

**현재 문제의 검증 과정:**
```
1. 요청:
   Origin: https://kotlip.kr
   credentials: 'include'

2. 응답:
   Access-Control-Allow-Origin: *
   Access-Control-Allow-Credentials: true

3. 브라우저 검증:
   ❌ "와일드카드와 credentials는 함께 사용 불가!"
   ❌ CORS 오류 발생
```

#### 4. 와일드카드와 credentials 충돌 감지

**브라우저의 검증 로직:**
```javascript
if (response.headers['Access-Control-Allow-Origin'] === '*' &&
    request.credentials === 'include') {
    throw new CORSException(
        "The value of the 'Access-Control-Allow-Origin' header " +
        "must not be the wildcard '*' when the request's " +
        "credentials mode is 'include'."
    );
}
```

**오류 메시지:**
```
Access to resource at 'https://proxy1.aiserv.ktcloud.com:10285/...' 
from origin 'https://kotlip.kr' has been blocked by CORS policy: 
The value of the 'Access-Control-Allow-Origin' header in the response 
must not be the wildcard '*' when the request's credentials mode is 'include'.
```

### 오류 메시지 해석

**오류 메시지 구성:**
1. **차단된 리소스**: `https://proxy1.aiserv.ktcloud.com:10285/...`
2. **요청 출처**: `from origin 'https://kotlip.kr'`
3. **차단 이유**: 와일드카드와 credentials 충돌

**의미:**
- 프론트엔드(`kotlip.kr`)에서 백엔드(`proxy1.aiserv.ktcloud.com`)로 직접 요청
- 백엔드가 `Access-Control-Allow-Origin: *` 응답
- 프론트엔드가 `credentials: 'include'` 사용
- 브라우저가 CORS 정책 위반으로 차단

---

## 5장: 현재 문제 단계별 분석

### 시나리오 1: 프론트엔드 → nginx → 백엔드 (정상)

#### 요청 흐름

```
[브라우저]                    [nginx]                    [백엔드]
   │                             │                           │
   │ 1. GET /api/video/...       │                           │
   │    Origin: kotlip.kr        │                           │
   │    credentials: include     │                           │
   ├────────────────────────────>│                           │
   │                             │                           │
   │                             │ 2. GET /api/video/...     │
   │                             │    Host: proxy1.aiserv... │
   │                             ├──────────────────────────>│
   │                             │                           │
   │                             │ 3. 응답                   │
   │                             │    Access-Control-...: *  │
   │                             │    (데이터)                │
   │                             │<──────────────────────────┤
   │                             │                           │
   │                             │ 4. nginx 헤더 처리        │
   │                             │    - 백엔드 헤더 숨김     │
   │                             │    - 새 헤더 추가         │
   │                             │                           │
   │ 5. 응답                     │                           │
   │    Access-Control-...:      │                           │
   │      https://kotlip.kr      │                           │
   │    Access-Control-...: true │                           │
   │    (데이터)                 │                           │
   │<────────────────────────────┤                           │
   │                             │                           │
   │ 6. 브라우저 CORS 검증 ✅    │                           │
   │    - Origin 일치 확인       │                           │
   │    - credentials 허용 확인  │                           │
   │                             │                           │
   │ 7. JavaScript에 응답 전달 ✅│                           │
```

#### 각 단계의 헤더 변화

**1단계: 브라우저 → nginx 요청**
```
GET /api/video/upload/status HTTP/1.1
Host: kotlip.kr
Origin: https://kotlip.kr
Cookie: session_id=abc123
```

**2단계: nginx → 백엔드 요청**
```
GET /api/video/upload/status HTTP/1.1
Host: proxy1.aiserv.ktcloud.com
X-Real-IP: 192.168.1.1
X-Forwarded-For: 192.168.1.1
X-Forwarded-Proto: https
```

**3단계: 백엔드 → nginx 응답**
```
HTTP/1.1 200 OK
Access-Control-Allow-Origin: *
Access-Control-Allow-Credentials: true
Content-Type: application/json

{"status": "uploading"}
```

**4단계: nginx 헤더 처리**
```nginx
proxy_hide_header Access-Control-Allow-Origin;        # * 제거
proxy_hide_header Access-Control-Allow-Credentials;    # true 제거
add_header Access-Control-Allow-Origin $http_origin;   # https://kotlip.kr 추가
add_header Access-Control-Allow-Credentials true;      # true 추가
```

**5단계: nginx → 브라우저 응답**
```
HTTP/1.1 200 OK
Access-Control-Allow-Origin: https://kotlip.kr
Access-Control-Allow-Credentials: true
Access-Control-Allow-Methods: GET, POST, PUT, PATCH, DELETE, OPTIONS
Access-Control-Allow-Headers: accept, authorization, content-type, ...
Content-Type: application/json

{"status": "uploading"}
```

#### 브라우저 검증 결과

**검증 단계:**
1. ✅ `Access-Control-Allow-Origin: https://kotlip.kr` 존재
2. ✅ 요청의 `Origin: https://kotlip.kr`와 일치
3. ✅ `Access-Control-Allow-Credentials: true` 존재
4. ✅ 와일드카드 없음 → credentials와 호환

**결과:** ✅ **검증 통과** → JavaScript에 응답 전달

### 시나리오 2: 프론트엔드 → 백엔드 직접 (문제 발생)

#### 요청 흐름

```
[브라우저]                    [nginx]                    [백엔드]
   │                             │                           │
   │ 1. GET https://proxy1...    │                           │
   │    Origin: kotlip.kr        │                           │
   │    credentials: include     │                           │
   │                             │                           │
   │                             │                           │
   │                             │                           │
   │────────────────────────────────────────────────────────>│
   │                             │                           │
   │                             │ 2. 응답                   │
   │                             │    Access-Control-...: *  │
   │                             │    Access-Control-...: true│
   │                             │    (데이터)                │
   │<────────────────────────────────────────────────────────┤
   │                             │                           │
   │ 3. 브라우저 CORS 검증 ❌    │                           │
   │    - Origin: kotlip.kr      │                           │
   │    - 응답: *                │                           │
   │    - credentials: include   │                           │
   │    - 와일드카드 + credentials│                           │
   │      충돌 감지!             │                           │
   │                             │                           │
   │ 4. CORS 오류 발생 ❌        │                           │
   │    응답 차단                │                           │
```

#### 백엔드 응답 헤더

**실제 확인된 응답:**
```
HTTP/1.1 200 OK
Access-Control-Allow-Origin: *
Access-Control-Allow-Credentials: true
Access-Control-Expose-Headers: Content-Type, X-CSRFToken
Content-Type: application/json

{"status": "uploading"}
```

**문제점:**
- `Access-Control-Allow-Origin: *` (와일드카드)
- `Access-Control-Allow-Credentials: true` (credentials 허용)
- 두 헤더가 동시에 존재 → CORS 정책 위반

#### 브라우저 검증 실패 원인

**검증 단계:**
1. ✅ `Access-Control-Allow-Origin: *` 존재
2. ✅ 요청의 `Origin: https://kotlip.kr` 확인
3. ✅ `Access-Control-Allow-Credentials: true` 존재
4. ❌ **와일드카드 `*`와 credentials `true` 동시 존재 → 즉시 실패**

**브라우저의 검증 로직:**
```javascript
if (response.headers['Access-Control-Allow-Origin'] === '*' &&
    response.headers['Access-Control-Allow-Credentials'] === 'true' &&
    request.credentials === 'include') {
    // CORS 정책 위반
    throw new CORSException("와일드카드와 credentials는 함께 사용 불가");
}
```

**결과:** ❌ **검증 실패** → 응답 차단, CORS 오류 발생

### 두 시나리오 비교표

| 항목 | 시나리오 1 (정상) | 시나리오 2 (문제) |
|------|------------------|------------------|
| **요청 경로** | `https://kotlip.kr/api/...` | `https://proxy1.aiserv.ktcloud.com:10285/api/...` |
| **nginx 거침** | ✅ 거침 | ❌ 우회 |
| **백엔드 응답 헤더** | `Access-Control-Allow-Origin: *` | `Access-Control-Allow-Origin: *` |
| **nginx 헤더 처리** | ✅ 백엔드 헤더 숨김, 새 헤더 추가 | ❌ 처리 없음 |
| **최종 응답 헤더** | `Access-Control-Allow-Origin: https://kotlip.kr` | `Access-Control-Allow-Origin: *` |
| **브라우저 검증** | ✅ 통과 | ❌ 실패 |
| **결과** | ✅ 정상 동작 | ❌ CORS 오류 |

**핵심 차이점:**
- 시나리오 1: nginx가 백엔드의 와일드카드 헤더를 특정 origin으로 교체
- 시나리오 2: 백엔드의 와일드카드 헤더가 그대로 브라우저에 전달

---

## 6장: 문제 해결 원리

### 왜 nginx를 거쳐야 하는가

#### nginx의 역할

nginx는 **CORS 헤더 변환기** 역할을 합니다:

1. **백엔드 헤더 숨기기**: 백엔드가 보낸 부적절한 CORS 헤더 제거
2. **올바른 헤더 추가**: 요청의 Origin을 확인하여 적절한 CORS 헤더 생성
3. **credentials 호환성**: 와일드카드 대신 특정 origin 사용으로 credentials 모드 지원

#### nginx를 거치지 않으면

**문제:**
- 백엔드가 `Access-Control-Allow-Origin: *` 응답
- 프론트엔드가 `credentials: 'include'` 사용
- 브라우저가 CORS 정책 위반으로 차단

**해결 불가능한 이유:**
- 백엔드 설정 변경이 어려운 경우
- 여러 프론트엔드 도메인 지원 필요
- 동적 origin 허용 필요

### nginx 설정의 의도된 동작

#### 설정 분석

```nginx
location /api {
    # 1. 백엔드로 프록시
    proxy_pass https://api_backend;
    
    # 2. 백엔드의 CORS 헤더 숨기기
    proxy_hide_header Access-Control-Allow-Origin;
    proxy_hide_header Access-Control-Allow-Credentials;
    
    # 3. 올바른 CORS 헤더 추가
    add_header Access-Control-Allow-Origin $http_origin always;
    add_header Access-Control-Allow-Credentials true always;
    add_header Access-Control-Allow-Methods "GET, POST, PUT, PATCH, DELETE, OPTIONS" always;
    add_header Access-Control-Allow-Headers "accept, authorization, content-type, user-agent, x-csrftoken, x-requested-with" always;
    
    # 4. Preflight 요청 처리
    if ($request_method = OPTIONS) {
        return 204;
    }
}
```

#### 동작 원리

**단계별 처리:**

1. **요청 수신**: `https://kotlip.kr/api/...` 요청 수신
2. **Origin 추출**: `$http_origin` 변수로 `Origin: https://kotlip.kr` 추출
3. **백엔드 프록시**: 요청을 백엔드로 전달
4. **응답 수신**: 백엔드로부터 `Access-Control-Allow-Origin: *` 포함 응답 수신
5. **헤더 변환**:
   - 백엔드의 `Access-Control-Allow-Origin: *` 제거
   - `Access-Control-Allow-Origin: https://kotlip.kr` 추가
6. **응답 전송**: 변환된 헤더와 함께 브라우저로 응답

**결과:**
- ✅ 와일드카드 제거
- ✅ 특정 origin 사용
- ✅ credentials 모드 지원
- ✅ 브라우저 CORS 검증 통과

### 프론트엔드 수정 방향

#### 현재 문제

**프론트엔드 코드 (추정):**
```javascript
// ❌ 잘못된 방식
const API_BASE_URL = 'https://proxy1.aiserv.ktcloud.com:10285';
fetch(`${API_BASE_URL}/api/video/upload/status/...`, {
  credentials: 'include'
});
```

**문제점:**
- nginx를 우회하여 백엔드로 직접 요청
- 백엔드의 와일드카드 헤더가 그대로 전달
- CORS 오류 발생

#### 올바른 수정 방향

**프론트엔드 코드 (수정):**
```javascript
// ✅ 올바른 방식
const API_BASE_URL = 'https://kotlip.kr';
fetch(`${API_BASE_URL}/api/video/upload/status/...`, {
  credentials: 'include'
});
```

**장점:**
- nginx를 거쳐 요청 전송
- nginx가 CORS 헤더를 올바르게 변환
- 브라우저 CORS 검증 통과
- credentials 모드 정상 동작

#### 수정 체크리스트

- [ ] API base URL을 `https://kotlip.kr`로 변경
- [ ] 모든 API 요청이 `/api` 경로로 시작하는지 확인
- [ ] `credentials: 'include'` 설정 유지 (인증 정보 전송 필요 시)
- [ ] 테스트: 브라우저 개발자 도구에서 CORS 오류 확인

**예상 결과:**
- ✅ 요청이 nginx를 거침
- ✅ nginx가 CORS 헤더 변환
- ✅ 브라우저 CORS 검증 통과
- ✅ 정상적으로 데이터 수신

---

## 결론

### 문제의 핵심

1. **프론트엔드가 nginx를 우회**하여 백엔드로 직접 요청
2. **백엔드가 와일드카드 `*` 응답** → credentials 모드와 충돌
3. **브라우저가 CORS 정책 위반 감지** → 요청 차단

### 해결 방법

**프론트엔드 API base URL 변경:**
- ❌ `https://proxy1.aiserv.ktcloud.com:10285`
- ✅ `https://kotlip.kr`

**nginx 설정은 이미 올바르게 구성되어 있음:**
- 백엔드 헤더 숨기기
- 동적 origin 허용
- credentials 모드 지원

### 최종 확인

nginx를 거친 요청의 응답 헤더 (실제 테스트 결과):
```
Access-Control-Allow-Origin: https://kotlip.kr
Access-Control-Allow-Credentials: true
Access-Control-Allow-Method


✅ **nginx 설정이 정상 동작함을 확인**

프론트엔드만 수정하면 CORS 오류가 해결됩니다.

---

## 부록: CORS 헤더 참고표

### 필수 CORS 헤더

| 헤더 | 의미 | 예시 |
|------|------|------|
| `Access-Control-Allow-Origin` | 허용할 출처 지정 | `https://kotlip.kr` 또는 `*` |
| `Access-Control-Allow-Credentials` | 인증 정보 전송 허용 | `true` |
| `Access-Control-Allow-Methods` | 허용할 HTTP 메서드 | `GET, POST, PUT, DELETE` |
| `Access-Control-Allow-Headers` | 허용할 요청 헤더 | `authorization, content-type` |

### CORS 헤더 조합 규칙

| Access-Control-Allow-Origin | Access-Control-Allow-Credentials | credentials: 'include' | 결과 |
|------------------------------|----------------------------------|------------------------|------|
| `*` | 없음 | 없음 | ✅ 허용 |
| `*` | `true` | `include` | ❌ **차단** |
| `https://kotlip.kr` | 없음 | 없음 | ✅ 허용 |
| `https://kotlip.kr` | `true` | `include` | ✅ 허용 |

### nginx 변수 참고

| 변수 | 의미 | 예시 |
|------|------|------|
| `$http_origin` | 요청의 Origin 헤더 값 | `https://kotlip.kr` |
| `$host` | 요청의 Host 헤더 값 | `kotlip.kr` |
| `$scheme` | 요청 프로토콜 | `https` |

---

## 용어 정리

- **Same-Origin Policy**: 동일 출처 정책, 브라우저의 기본 보안 정책
- **Cross-Origin**: 다른 출처, 프로토콜/도메인/포트 중 하나라도 다른 경우
- **CORS**: Cross-Origin Resource Sharing, Cross-Origin 요청을 허용하는 메커니즘
- **Preflight**: 실제 요청 전에 서버의 CORS 정책을 확인하는 OPTIONS 요청
- **credentials**: 인증 정보(쿠키, Authorization 헤더 등)를 포함한 요청
- **와일드카드**: `*` 기호로 모든 것을 허용하는 표현

---

## 참고 자료

- [MDN Web Docs - CORS](https://developer.mozilla.org/ko/docs/Web/HTTP/CORS)
- [W3C CORS Specification](https://www.w3.org/TR/cors/)
- [nginx add_header 문서](http://nginx.org/en/docs/http/ngx_http_headers_module.html#add_header)